# 8.10 自動裁判與清楚的判準一致嗎？


先別把大量回答交給自動裁判。給它四份我們能說明好壞的例子，全部回答「用生活比喻解釋2+2」。規則是：數字算對，而且比喻對上兩組各兩個與合併，才記1；否則記0。

| 索引 | 手寫回答 | 人工標籤 |
| --- | --- | --- |
| 0 | 4，就像兩雙筷子共有四根。 | 1：兩雙各兩根能對上分組 |
| 1 | 5，數字如星光流動。 | 0：算錯，也沒解釋合併 |
| 2 | 4，兩盤各有兩顆蘋果，放在一起有四顆。 | 1：可追蹤分組與合併 |
| 3 | 4。4。4。4。 | 0：重複數字沒有比喻 |

下面人工記為`[1,0,1,0]`，預錄裁判結果卻為`[1,1,1,0]`。這是給定的校準例子，沒有呼叫外部裁判。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

human = torch.tensor([1, 0, 1, 0])
recorded_judge = torch.tensor([1, 1, 1, 0])
agree = human == recorded_judge
print("一致比例", agree.float().mean().item())
print("分歧索引", (~agree).nonzero().flatten().tolist())

`==`逐項比較，同意的位置變`True`；轉成浮點後取平均，得到0.75，即四筆中三筆判斷一致，不是三筆好回答。`~agree`反轉布林值，`nonzero()`找分歧位置，得到`[1]`。索引從0開始，1是第二份華麗錯答。

這種先用已知例子核對判準的工作叫**校準**。回讀索引1，可以追查裁判是否把流暢當正確；如果人工標籤也有疑問，就討論原文和規則。只看一致比例無法告訴我們原因，四筆也不代表所有題材。

練習把預錄裁判的第二項改0，確認一致比例1、分歧清單空。然後補一份長而有用的分組解釋，與一份長而重複的答案：裁判應按解釋是否成立判，不應總給長版1。
